# Module 06 Lab — LangGraph: Multi-Agent Research Pipeline

**Course:** AI Agents Teaching Platform  
**Module:** 06 — Multi-Agent Systems  
**Estimated time:** ~90 minutes

---

### What you'll build

A 4-agent research pipeline using LangGraph's directed graph model:
- **Orchestrator** decomposes the question into two sub-questions
- **Compression Specialist** researches model compression angle
- **Inference Specialist** researches inference optimisation angle
- **Critic** reviews both outputs; triggers repair if quality < threshold

> **Note:** This lab uses `langchain-anthropic` for the LLM client since LangGraph is tightly integrated with LangChain. Set `ANTHROPIC_API_KEY` in Step 3.

In [ ]:
%pip install -q langgraph langchain-anthropic

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("Key set ✓")

## Step 1 — Define the shared state

The state schema is the contract between all agents. All nodes read from and write to this shared dict.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic
import json

llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

class ResearchState(TypedDict):
    question: str
    sub_question_a: str          # model compression angle
    sub_question_b: str          # inference optimisation angle
    specialist_a_output: dict    # {findings: [...], key_claim: '...'}
    specialist_b_output: dict    # {findings: [...], key_claim: '...'}
    critic_verdict: dict         # {pass: bool, score: float, issues: [...]}
    repair_count: int
    final_answer: str

print("State schema defined ✓")

## Step 2 — Define the agent nodes (TODO)

Each node is a function that receives `state: ResearchState` and returns a dict of state updates.

**TODO:** Fill in `orchestrator_decompose` so it calls the LLM and returns
`{"sub_question_a": ..., "sub_question_b": ..., "repair_count": 0}`.

The specialist and critic nodes are complete — read them carefully.

In [ ]:
def orchestrator_decompose(state: ResearchState) -> dict:
    """TODO: call llm.invoke([...]) asking for JSON with sub_question_a and sub_question_b."""
    # Hint: use [{"role": "system", "content": "Decompose the question..."},
    #            {"role": "user", "content": state["question"]}]
    # Return json.loads(response.content) merged with {"repair_count": 0}
    pass


def specialist_compression(state: ResearchState) -> dict:
    response = llm.invoke([
        {"role": "system", "content": (
            "Research model compression techniques (quantisation, pruning, distillation). "
            "Return JSON: {\"findings\": [\"finding 1\", ...], \"key_claim\": \"...\"}" 
        )},
        {"role": "user", "content": state["sub_question_a"]}
    ])
    try:
        return {"specialist_a_output": json.loads(response.content)}
    except json.JSONDecodeError:
        return {"specialist_a_output": {"findings": [response.content], "key_claim": ""}}


def specialist_inference(state: ResearchState) -> dict:
    response = llm.invoke([
        {"role": "system", "content": (
            "Research inference optimisation (speculative decoding, KV cache, continuous batching). "
            "Return JSON: {\"findings\": [\"finding 1\", ...], \"key_claim\": \"...\"}"
        )},
        {"role": "user", "content": state["sub_question_b"]}
    ])
    try:
        return {"specialist_b_output": json.loads(response.content)}
    except json.JSONDecodeError:
        return {"specialist_b_output": {"findings": [response.content], "key_claim": ""}}


def critic_review(state: ResearchState) -> dict:
    content = (
        f"Specialist A (compression): {json.dumps(state['specialist_a_output'])}\n"
        f"Specialist B (inference): {json.dumps(state['specialist_b_output'])}"
    )
    response = llm.invoke([
        {"role": "system", "content": (
            "Evaluate both research outputs. Return JSON: "
            "{\"pass\": true/false, \"score\": 0.0-1.0, \"issues\": [\"...\"]}"
        )},
        {"role": "user", "content": content}
    ])
    try:
        verdict = json.loads(response.content)
    except json.JSONDecodeError:
        verdict = {"pass": True, "score": 0.5, "issues": []}
    return {"critic_verdict": verdict, "repair_count": state["repair_count"] + 1}


def synthesise(state: ResearchState) -> dict:
    response = llm.invoke([{
        "role": "user",
        "content": (
            f"Synthesise these research findings into a 300-word answer:\n\n"
            f"Question: {state['question']}\n"
            f"Compression findings: {state['specialist_a_output']}\n"
            f"Inference findings: {state['specialist_b_output']}\n"
            f"Critic notes: {state['critic_verdict'].get('issues', [])}"
        )
    }])
    return {"final_answer": response.content}


print("Agent nodes defined ✓")

<details>
<summary>💡 Stuck? Reveal orchestrator_decompose solution</summary>

```python
def orchestrator_decompose(state: ResearchState) -> dict:
    response = llm.invoke([
        {"role": "system", "content": (
            "Decompose the research question into two sub-questions: "
            "one focused on model compression, one on inference optimisation. "
            'Return JSON: {"sub_question_a": "...", "sub_question_b": "..."}'
        )},
        {"role": "user", "content": state["question"]}
    ])
    try:
        parsed = json.loads(response.content)
    except json.JSONDecodeError:
        parsed = {"sub_question_a": state["question"], "sub_question_b": state["question"]}
    return {**parsed, "repair_count": 0}
```

</details>

## Step 3 — Build the graph (TODO)

Wire the nodes and edges. Key decisions:
- After decompose: both specialists run (fan-out)
- After both specialists: critic reviews
- After critic: conditional edge — if `pass` or `repair_count >= 2`, go to `synthesise`; else go back to `specialist_a` for repair

**TODO:** Implement `route_after_critic` and add the conditional edge.

In [ ]:
def route_after_critic(state: ResearchState) -> str:
    """TODO: return 'synthesise' if verdict passes or repair_count >= 2, else return 'repair'."""
    # state["critic_verdict"].get("pass", False)
    # state["repair_count"]
    pass


graph = StateGraph(ResearchState)

graph.add_node("decompose",   orchestrator_decompose)
graph.add_node("specialist_a", specialist_compression)
graph.add_node("specialist_b", specialist_inference)
graph.add_node("critic",      critic_review)
graph.add_node("synthesise",  synthesise)

graph.set_entry_point("decompose")
graph.add_edge("decompose",    "specialist_a")
graph.add_edge("decompose",    "specialist_b")
graph.add_edge("specialist_a", "critic")
graph.add_edge("specialist_b", "critic")

# TODO: add_conditional_edges from "critic" using route_after_critic
# {"synthesise": "synthesise", "repair": "specialist_a"}

graph.add_edge("synthesise", END)

pipeline = graph.compile(checkpointer=MemorySaver())
print("Graph compiled ✓")

<details>
<summary>💡 Stuck? Reveal the routing and conditional edge solution</summary>

```python
def route_after_critic(state: ResearchState) -> str:
    if state["critic_verdict"].get("pass", False) or state["repair_count"] >= 2:
        return "synthesise"
    return "repair"

graph.add_conditional_edges("critic", route_after_critic,
    {"synthesise": "synthesise", "repair": "specialist_a"})
```

</details>

## Step 4 — Run the pipeline

In [ ]:
QUESTION = (
    "What is the current state of LLM efficiency research — "
    "specifically model compression and inference optimisation?"
)

config = {"configurable": {"thread_id": "research-001"}}
result = pipeline.invoke({"question": QUESTION}, config=config)

print("Sub-questions:")
print(f"  A: {result['sub_question_a']}")
print(f"  B: {result['sub_question_b']}")
print(f"\nCritic verdict: {result['critic_verdict']}")
print(f"\nFinal answer ({len(result['final_answer'])} chars):")
print(result["final_answer"])

## Step 5 — Inspect the checkpoint

LangGraph checkpoints let you resume a paused pipeline without re-running completed stages.

In [ ]:
# The MemorySaver keeps state in memory — inspect it
state_snapshot = pipeline.get_state(config)
print("Checkpointed state keys:", list(state_snapshot.values.keys()))
print("Repair count:", state_snapshot.values.get("repair_count"))
print("Critic score:", state_snapshot.values.get("critic_verdict", {}).get("score"))

## Step 6 — Exercises

**Exercise 1: interrupt_before the critic**
```python
pipeline = graph.compile(checkpointer=MemorySaver(), interrupt_before=["critic"])
```
Run the pipeline. It will pause before the critic. Inspect the state,
then call `pipeline.invoke(None, config=config)` to resume.

**Exercise 2: Add a fourth node**
Add a `fact_checker` node between `critic` and `synthesise`. It should verify that
every claim in the specialist outputs includes a citation (string containing '[' or 'Author').

**Exercise 3: Measure token cost**
Accumulate `response.usage_metadata` across all node calls. How many tokens does the full pipeline use?
Compare to a single LLM call on the same question.

**Exercise 4: Change the LLM**
LangGraph supports any LangChain-compatible LLM. Try `ChatOpenAI(model='gpt-4o-mini')` (requires `langchain-openai`).
Does the graph structure change? (Answer: no — only the `llm = ...` line changes.)

In [ ]:
# Scratch cell — use for exercises


## Step 7 — Reflection

1. What is the key advantage of TypedDict state vs passing data as function arguments?
2. Why does `route_after_critic` cap repairs at 2 (`repair_count >= 2`)?
3. What does `interrupt_before` give you that a plain conditional can't?
4. If you replaced `MemorySaver` with `PostgresSaver`, what changes? What stays the same?

*(Double-click to edit)*

1. 
2. 
3. 
4. 